# 02 SOKE20FPS NB51 Tokenizer Colab

Train a  `t08_s32_cd1024`  tokenizer on the new SOKE_DATA datasets.

This notebook expects the neighboring `scripts/` directory to travel with it. Neural Sign Actors is treated as the How2Sign source. The preprocessing step downsamples to 20 fps and applies SOKE-style clip rejection before building the Notebook-51 canonical cache.

In [ ]:
# Run configuration: edit this cell first in Colab.
from pathlib import Path
import os
import shlex
import subprocess
import sys

DRIVE_ROOT = Path(os.environ.get(
    'SOKE20_DRIVE_ROOT',
    '/content/drive/MyDrive/folder/COLAB',
))
SOKE_DATA_ROOT_ENV = os.environ.get('SOKE_DATA_ROOT')
SOKE20_INDEX_CSV_ENV = os.environ.get('SOKE20_INDEX_CSV')
SOKE20_OUTPUT_ROOT_ENV = os.environ.get('SOKE20_OUTPUT_ROOT')
SOKE20_RUN_NAME_ENV = os.environ.get('SOKE20_RUN_NAME')
SMPLX_MODEL_ROOT_ENV = os.environ.get('SMPLX_MODEL_ROOT')
DATA_ZIP_ROOT_ENV = os.environ.get('SOKE20_DATA_ZIP_ROOT')
DATA_EXTRACT_ROOT_ENV = os.environ.get('SOKE20_DATA_EXTRACT_ROOT')
ALLOW_DRIVE_EXTRACTED_DATA = os.environ.get('SOKE20_ALLOW_DRIVE_EXTRACTED_DATA', '0') == '1'

TARGET_FPS = float(os.environ.get('SOKE20_TARGET_FPS', '20'))
WINDOW_SIZE = int(os.environ.get('SOKE20_WINDOW_SIZE', '64'))
MAX_CLIPS_PER_SPLIT = int(os.environ.get('SOKE20_MAX_CLIPS_PER_SPLIT', '0'))  # set 2-16 for smoke
PREPROCESS_FORCE = os.environ.get('SOKE20_PREPROCESS_FORCE', '0') == '1'
AUTO_EXTRACT_DATA = os.environ.get('SOKE20_AUTO_EXTRACT_DATA', '1') == '1'
DATA_EXTRACT_FORCE = os.environ.get('SOKE20_DATA_EXTRACT_FORCE', '0') == '1'
DATA_EXTRACT_WORKERS = int(os.environ.get('SOKE20_DATA_EXTRACT_WORKERS', '2'))
DATA_EXTRACTOR = os.environ.get('SOKE20_DATA_EXTRACTOR', 'auto')

EPOCHS = int(os.environ.get('SOKE20_EPOCHS', '300'))
BATCH_SIZE = int(os.environ.get('SOKE20_BATCH_SIZE', '32'))
GRAD_ACCUM = int(os.environ.get('SOKE20_GRAD_ACCUM', '1'))
NUM_WORKERS = int(os.environ.get('SOKE20_NUM_WORKERS', '2'))
PRELOAD = int(os.environ.get('SOKE20_PRELOAD', '0'))
QUANTIZER_FRAME_CAP = int(os.environ.get('SOKE20_QUANTIZER_FRAME_CAP', '200000'))
REBUILD_QUANTIZER = os.environ.get('SOKE20_REBUILD_QUANTIZER', '0') == '1'
RUN_TRAINING = os.environ.get('SOKE20_RUN_TRAINING', '1') == '1'
RESUME_TRAINING = int(os.environ.get('SOKE20_RESUME_TRAINING', '1'))

# NB51 t08_s32 geometry, with the requested coarser scalar target.
D_MODEL = int(os.environ.get('SOKE20_D_MODEL', '256'))
CODE_DIM = int(os.environ.get('SOKE20_CODE_DIM', '1024'))
CODE_NUM = int(os.environ.get('SOKE20_CODE_NUM', '1024'))
NUM_BINS = int(os.environ.get('SOKE20_NUM_BINS', '128'))

RUN_VALIDATION_METRICS = os.environ.get('SOKE20_RUN_VALIDATION_METRICS', '1') == '1'
VALIDATION_METRIC_SPLIT = os.environ.get('SOKE20_VALIDATION_METRIC_SPLIT', 'val')
VALIDATION_METRIC_MAX_CLIPS_PER_DATASET = int(os.environ.get('SOKE20_VALIDATION_METRIC_MAX_CLIPS_PER_DATASET', '0'))
VALIDATION_METRIC_STRIDE_SIZE = int(os.environ.get('SOKE20_VALIDATION_METRIC_STRIDE_SIZE', '32'))
VALIDATION_METRIC_COMPUTE_DTW_PA = int(os.environ.get('SOKE20_VALIDATION_METRIC_COMPUTE_DTW_PA', '1'))
VALIDATION_METRIC_FORCE = os.environ.get('SOKE20_VALIDATION_METRIC_FORCE', '0') == '1'

SYNC_TO_DRIVE = os.environ.get('SOKE20_SYNC_TO_DRIVE', '1') != '0'
DRIVE_SYNC_RETRIES = int(os.environ.get('SOKE20_DRIVE_SYNC_RETRIES', '4'))
DRIVE_SYNC_RETRY_SLEEP_SEC = float(os.environ.get('SOKE20_DRIVE_SYNC_RETRY_SLEEP_SEC', '20'))
DRIVE_SYNC_DEST = Path(os.environ.get(
    'SOKE20_DRIVE_SYNC_DEST',
    str(DRIVE_ROOT / 'Tokenizer' / 'outputs'),
))

print({
    'DRIVE_ROOT': str(DRIVE_ROOT),
    'TARGET_FPS': TARGET_FPS,
    'WINDOW_SIZE': WINDOW_SIZE,
    'MAX_CLIPS_PER_SPLIT': MAX_CLIPS_PER_SPLIT,
    'AUTO_EXTRACT_DATA': AUTO_EXTRACT_DATA,
    'DATA_EXTRACT_FORCE': DATA_EXTRACT_FORCE,
    'ALLOW_DRIVE_EXTRACTED_DATA': ALLOW_DRIVE_EXTRACTED_DATA,
    'DATA_EXTRACT_WORKERS': DATA_EXTRACT_WORKERS,
    'EPOCHS': EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'GRAD_ACCUM': GRAD_ACCUM,
    'NUM_WORKERS': NUM_WORKERS,
    'D_MODEL': D_MODEL,
    'CODE_DIM': CODE_DIM,
    'CODE_NUM': CODE_NUM,
    'NUM_BINS': NUM_BINS,
    'RESUME_TRAINING': RESUME_TRAINING,
    'RUN_VALIDATION_METRICS': RUN_VALIDATION_METRICS,
    'VALIDATION_METRIC_SPLIT': VALIDATION_METRIC_SPLIT,
    'VALIDATION_METRIC_MAX_CLIPS_PER_DATASET': VALIDATION_METRIC_MAX_CLIPS_PER_DATASET,
    'VALIDATION_METRIC_COMPUTE_DTW_PA': VALIDATION_METRIC_COMPUTE_DTW_PA,
    'SYNC_TO_DRIVE': SYNC_TO_DRIVE,
    'DRIVE_SYNC_DEST': str(DRIVE_SYNC_DEST),
})

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and os.environ.get('SOKE20_SKIP_DRIVE_MOUNT', '0') != '1':
    drive.mount('/content/drive')

# If opened directly from Drive, current working directory should be this notebook folder.
THIS_DIR = Path.cwd().resolve()
if not (THIS_DIR / 'scripts').exists():
    # Common fallbacks for local execution, /content startup, and the Tokenizer Drive bundle.
    candidates = [
        Path.cwd() / 'new_data' / '02_soke20fps_nb51_tokenizer',
        DRIVE_ROOT / 'Tokenizer',
        DRIVE_ROOT / 'llm_fine_tune_colab' / 'new_data' / '02_soke20fps_nb51_tokenizer',
        Path('/content/drive/MyDrive/folder/COLAB/Tokenizer'),
    ]
    for c in candidates:
        if (c / 'scripts').exists():
            THIS_DIR = c.resolve()
            os.chdir(THIS_DIR)
            break

if not (THIS_DIR / 'scripts').exists():
    raise FileNotFoundError(f'Could not find scripts/ next to notebook. cwd={Path.cwd()}')

sys.path.insert(0, str(THIS_DIR / 'scripts'))
print('THIS_DIR =', THIS_DIR)
print('Python =', sys.executable)

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(torch.cuda.current_device()))


In [ ]:
def has_soke_data_root(path: Path | None) -> bool:
    if path is None:
        return False
    return (
        (path / 'CSL-Daily-Fittings').exists()
        and (path / 'Neural-Sign-Actors').exists()
        and (path / 'phoenix_poses').exists()
    )


def has_top_level_zips(path: Path | None) -> bool:
    if path is None:
        return False
    return all((path / name).exists() for name in [
        'CSL-Daily-Fittings.zip',
        'Neural-Sign-Actors.zip',
        'phoenix_poses.zip',
    ])

DATA_CANDIDATES = [
    Path(SOKE_DATA_ROOT_ENV) if SOKE_DATA_ROOT_ENV else None,
    Path('/content/SOKE_DATA'),
    Path('/content/DATA/SOKE_DATA'),
    Path('/content/DATA'),
]
if ALLOW_DRIVE_EXTRACTED_DATA:
    DATA_CANDIDATES += [
        THIS_DIR.parent / 'DATA',
        DRIVE_ROOT / 'DATA',
        DRIVE_ROOT / 'DATA' / 'SOKE_DATA',
        THIS_DIR.parent.parent / 'DATA' / 'SOKE_DATA',
        THIS_DIR.parent.parent / 'DATA',
    ]
SOKE_DATA_ROOT = next((p for p in DATA_CANDIDATES if has_soke_data_root(p)), None)

ZIP_CANDIDATES = [
    Path(DATA_ZIP_ROOT_ENV) if DATA_ZIP_ROOT_ENV else None,
    THIS_DIR.parent / 'DATA' / 'ZIPS',
    DRIVE_ROOT / 'DATA' / 'ZIPS',
    Path('/content/drive/MyDrive/folder/COLAB/DATA/ZIPS'),
]
DATA_ZIP_ROOT = next((p for p in ZIP_CANDIDATES if has_top_level_zips(p)), None)

if SOKE_DATA_ROOT is None and AUTO_EXTRACT_DATA:
    if DATA_ZIP_ROOT is None:
        print('No extracted runtime data root and no DATA/ZIPS folder found. Set SOKE_DATA_ROOT or SOKE20_DATA_ZIP_ROOT.')
    else:
        extract_root = Path(DATA_EXTRACT_ROOT_ENV).resolve() if DATA_EXTRACT_ROOT_ENV else (
            Path('/content/SOKE_DATA') if IN_COLAB else Path('/tmp/SOKE_DATA')
        )
        script_candidates = [
            THIS_DIR / 'scripts' / 'unpack_recursive_zips.py',
            DATA_ZIP_ROOT / 'unpack_recursive_zips.py',
        ]
        unpack_script = next((p for p in script_candidates if p.exists()), None)
        if unpack_script is None:
            raise FileNotFoundError('Could not find unpack_recursive_zips.py in scripts/ or DATA/ZIPS.')
        cmd = [
            sys.executable,
            str(unpack_script),
            str(DATA_ZIP_ROOT),
            '--output-root', str(extract_root),
            '--workers', str(DATA_EXTRACT_WORKERS),
            '--extractor', str(DATA_EXTRACTOR),
        ]
        if DATA_EXTRACT_FORCE:
            cmd.append('--force')
        print(' '.join(shlex.quote(x) for x in cmd))
        subprocess.run(cmd, check=True)
        if has_soke_data_root(extract_root):
            SOKE_DATA_ROOT = extract_root

if SOKE_DATA_ROOT is None:
    raise FileNotFoundError(
        'Could not resolve runtime SOKE data. Expected a root containing CSL-Daily-Fittings, '
        'Neural-Sign-Actors, and phoenix_poses. By default the notebook extracts zips from Drive '
        'to /content/SOKE_DATA. Set SOKE_DATA_ROOT only for an already-extracted runtime/local path.'
    )

INDEX_CANDIDATES = [
    Path(SOKE20_INDEX_CSV_ENV) if SOKE20_INDEX_CSV_ENV else None,
    THIS_DIR / 'combined_clip_index.csv',
    THIS_DIR.parent / '01_csl_phoenix_neural_sign_actors_exploration_and_stats' / 'combined_clip_index.csv',
    DRIVE_ROOT / 'llm_fine_tune_colab' / 'new_data' / '01_csl_phoenix_neural_sign_actors_exploration_and_stats' / 'combined_clip_index.csv',
]
INDEX_CSV = next((p for p in INDEX_CANDIDATES if p is not None and p.exists()), None)


def has_smplx_model(path: Path | None) -> bool:
    if path is None:
        return False
    return (
        (path / 'smplx' / 'SMPLX_NEUTRAL.npz').exists()
        or (path / 'smplx' / 'SMPLX_NEUTRAL.pkl').exists()
        or (path / 'SMPLX_NEUTRAL.npz').exists()
        or (path / 'SMPLX_NEUTRAL.pkl').exists()
    )

BODY_MODEL_CANDIDATES = [
    Path(SMPLX_MODEL_ROOT_ENV) if SMPLX_MODEL_ROOT_ENV else None,
    THIS_DIR.parent / 'body_models',
    THIS_DIR.parent.parent / 'body_models',
    DRIVE_ROOT / 'body_models',
    Path('/content/body_models'),
]
BODY_MODEL_ROOT = next((p for p in BODY_MODEL_CANDIDATES if has_smplx_model(p)), None)

DEFAULT_OUTPUT_ROOT = Path('/content/soke20_nb51_tokenizer_outputs') if IN_COLAB else (THIS_DIR / 'outputs')
OUTPUT_ROOT = Path(SOKE20_OUTPUT_ROOT_ENV).resolve() if SOKE20_OUTPUT_ROOT_ENV else DEFAULT_OUTPUT_ROOT.resolve()
RUN_NAME = SOKE20_RUN_NAME_ENV or f't08_s32_cd{CODE_DIM}_cn{CODE_NUM}_b{BATCH_SIZE}_ga{GRAD_ACCUM}_soke20'
PREPROCESS_ROOT = OUTPUT_ROOT / 'preprocess_soke20'
RUN_ROOT = OUTPUT_ROOT / 'runs' / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('SOKE_DATA_ROOT =', SOKE_DATA_ROOT)
print('DATA_ZIP_ROOT =', DATA_ZIP_ROOT)
print('INDEX_CSV =', INDEX_CSV)
print('BODY_MODEL_ROOT =', BODY_MODEL_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('PREPROCESS_ROOT =', PREPROCESS_ROOT)
print('RUN_NAME =', RUN_NAME)
print('RUN_ROOT =', RUN_ROOT)

In [ ]:
cmd = [
    sys.executable,
    str(THIS_DIR / 'scripts' / 'soke20_preprocess.py'),
    '--data-root', str(SOKE_DATA_ROOT),
    '--out-root', str(PREPROCESS_ROOT),
    '--target-fps', str(TARGET_FPS),
    '--window-size', str(WINDOW_SIZE),
    '--progress-every', '250',
]
if INDEX_CSV is not None:
    cmd += ['--index-csv', str(INDEX_CSV)]
if MAX_CLIPS_PER_SPLIT > 0:
    cmd += ['--max-clips-per-split', str(MAX_CLIPS_PER_SPLIT)]
if PREPROCESS_FORCE:
    cmd += ['--force']

print(' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)


In [ ]:
import pandas as pd
summary_path = PREPROCESS_ROOT / 'preprocess_summary.csv'
manifest_path = PREPROCESS_ROOT / 'manifest_all.csv'
print('summary_path =', summary_path)
print('manifest_path =', manifest_path)
summary = pd.read_csv(summary_path)
display(summary)
print('OK rows:', int((pd.read_csv(manifest_path)['status'] == 'ok').sum()))


In [ ]:
if RUN_TRAINING:
    cmd = [
        sys.executable,
        str(THIS_DIR / 'scripts' / 'train_nb51_soke20.py'),
        '--preprocess-root', str(PREPROCESS_ROOT),
        '--run-root', str(RUN_ROOT),
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--grad-accum', str(GRAD_ACCUM),
        '--num-workers', str(NUM_WORKERS),
        '--preload', str(PRELOAD),
        '--resume', str(RESUME_TRAINING),
        '--quantizer-frame-cap', str(QUANTIZER_FRAME_CAP),
        '--num-bins', str(NUM_BINS),
        '--d-model', str(D_MODEL),
        '--code-dim', str(CODE_DIM),
        '--code-num', str(CODE_NUM),
    ]
    if REBUILD_QUANTIZER:
        cmd += ['--rebuild-quantizer']
    print(' '.join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, check=True)
else:
    print('Training skipped because SOKE20_RUN_TRAINING=0')


In [ ]:
from IPython.display import Image, display
hist_path = RUN_ROOT / 'history.csv'
summary_path = RUN_ROOT / 'results_summary.csv'
plot_path = RUN_ROOT / 'plots' / 'training_curves.png'
print('history =', hist_path)
print('summary =', summary_path)
if hist_path.exists():
    hist = pd.read_csv(hist_path)
    display(hist.tail())
if summary_path.exists():
    display(pd.read_csv(summary_path))
if plot_path.exists():
    display(Image(filename=str(plot_path)))


In [ ]:
# Per-dataset validation metrics: MPJPE/JPE, PA-MPJPE, DTW-MPJPE, and optional DTW-PA-MPJPE.
if RUN_VALIDATION_METRICS:
    if BODY_MODEL_ROOT is None:
        print('Validation metrics skipped: set SMPLX_MODEL_ROOT to a folder containing smplx/SMPLX_NEUTRAL.npz.')
    elif not (RUN_ROOT / 'best.pt').exists():
        print('Validation metrics skipped: best checkpoint not found:', RUN_ROOT / 'best.pt')
    else:
        cmd = [
            sys.executable,
            str(THIS_DIR / 'scripts' / 'eval_nb51_soke20_metrics.py'),
            '--preprocess-root', str(PREPROCESS_ROOT),
            '--run-root', str(RUN_ROOT),
            '--body-model-root', str(BODY_MODEL_ROOT),
            '--split', str(VALIDATION_METRIC_SPLIT),
            '--max-clips-per-dataset', str(VALIDATION_METRIC_MAX_CLIPS_PER_DATASET),
            '--stride-size', str(VALIDATION_METRIC_STRIDE_SIZE),
            '--compute-dtw-pa', str(VALIDATION_METRIC_COMPUTE_DTW_PA),
        ]
        if VALIDATION_METRIC_FORCE:
            cmd += ['--force']
        print(' '.join(shlex.quote(x) for x in cmd))
        subprocess.run(cmd, check=True)

        suffix = f'{VALIDATION_METRIC_SPLIT}_fullclip_per_dataset_motion'
        if VALIDATION_METRIC_MAX_CLIPS_PER_DATASET > 0:
            suffix += f'_limit{VALIDATION_METRIC_MAX_CLIPS_PER_DATASET}'
        metric_summary_path = RUN_ROOT / f'{suffix}_summary.csv'
        metric_plot_path = RUN_ROOT / 'plots' / f'{suffix}_summary.png'
        print('metric_summary =', metric_summary_path)
        if metric_summary_path.exists():
            display(pd.read_csv(metric_summary_path))
        if metric_plot_path.exists():
            display(Image(filename=str(metric_plot_path)))
else:
    print('Validation metrics skipped because SOKE20_RUN_VALIDATION_METRICS=0')

In [ ]:
# Robust final Google Drive save. Runs after training/eval and before runtime cleanup.
import json
import shutil
import time
from datetime import datetime, timezone

metric_suffix = f'{VALIDATION_METRIC_SPLIT}_fullclip_per_dataset_motion'
if VALIDATION_METRIC_MAX_CLIPS_PER_DATASET > 0:
    metric_suffix += f'_limit{VALIDATION_METRIC_MAX_CLIPS_PER_DATASET}'

run_sync_files = [
    'best.pt',
    'last.pt',
    'history.csv',
    'results_summary.csv',
    'train_args.json',
    'train_args.requested.json',
    'quantizer_stats.npz',
    f'{metric_suffix}_metrics.csv',
    f'{metric_suffix}_summary.csv',
    f'{metric_suffix}_summary.json',
]
run_file_sizes = {name: (RUN_ROOT / name).stat().st_size for name in run_sync_files if (RUN_ROOT / name).exists()}

final_status = {
    'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    'this_dir': str(THIS_DIR),
    'output_root': str(OUTPUT_ROOT),
    'preprocess_root': str(PREPROCESS_ROOT),
    'run_name': str(RUN_ROOT.name),
    'run_root': str(RUN_ROOT),
    'history_csv': str(RUN_ROOT / 'history.csv'),
    'results_summary_csv': str(RUN_ROOT / 'results_summary.csv'),
    'metric_summary_csv': str(RUN_ROOT / f'{metric_suffix}_summary.csv'),
    'metric_summary_json': str(RUN_ROOT / f'{metric_suffix}_summary.json'),
    'metric_plot': str(RUN_ROOT / 'plots' / f'{metric_suffix}_summary.png'),
    'best_ckpt': str(RUN_ROOT / 'best.pt'),
    'last_ckpt': str(RUN_ROOT / 'last.pt'),
    'num_bins': NUM_BINS,
    'batch_size': BATCH_SIZE,
    'grad_accum': GRAD_ACCUM,
    'run_validation_metrics': RUN_VALIDATION_METRICS,
    'validation_metric_split': VALIDATION_METRIC_SPLIT,
    'validation_metric_max_clips_per_dataset': VALIDATION_METRIC_MAX_CLIPS_PER_DATASET,
    'sync_to_drive': bool(SYNC_TO_DRIVE),
    'drive_sync_dest': str(DRIVE_SYNC_DEST),
    'drive_run_sync_dest': str(DRIVE_SYNC_DEST / 'runs' / RUN_ROOT.name),
    'run_file_sizes': run_file_sizes,
}
status_path = OUTPUT_ROOT / 'final_save_status.json'
status_path.parent.mkdir(parents=True, exist_ok=True)
status_path.write_text(json.dumps(final_status, indent=2), encoding='utf-8')
print('Wrote final status:', status_path)


def same_resolved_path(a: Path, b: Path) -> bool:
    try:
        return a.resolve() == b.resolve()
    except Exception:
        return False


def robust_rsync(
    src: Path,
    dst: Path,
    retries: int,
    sleep_sec: float,
    *,
    verify_rel: str | None = None,
    extra_args: list[str] | None = None,
) -> None:
    src = src.resolve()
    dst = dst.resolve()
    dst.mkdir(parents=True, exist_ok=True)
    cmd = ['rsync', '-a', '--info=progress2']
    if extra_args:
        cmd.extend(extra_args)
    cmd.extend([str(src) + '/', str(dst) + '/'])
    last_error = None
    for attempt in range(1, int(retries) + 1):
        print(f'Drive sync attempt {attempt}/{retries}:', ' '.join(shlex.quote(x) for x in cmd))
        result = subprocess.run(cmd, text=True)
        if result.returncode == 0:
            if verify_rel is None or (dst / verify_rel).exists():
                print('Drive sync verified:', dst)
                return
            last_error = RuntimeError(f'rsync finished but {verify_rel} was not visible at destination')
        else:
            last_error = RuntimeError(f'rsync failed with returncode={result.returncode}')
        if attempt < int(retries):
            print(f'Sync failed; retrying in {sleep_sec} seconds:', last_error)
            time.sleep(float(sleep_sec))
    raise RuntimeError(f'Final Drive sync failed after {retries} attempts: {last_error}')


def verify_synced_files(src_dir: Path, dst_dir: Path, file_sizes: dict[str, int]) -> None:
    missing_or_bad = []
    for rel, expected_size in file_sizes.items():
        src = src_dir / rel
        dst = dst_dir / rel
        if not src.exists():
            continue
        if not dst.exists():
            missing_or_bad.append(f'{rel}: missing')
            continue
        actual_size = dst.stat().st_size
        if int(actual_size) != int(expected_size):
            missing_or_bad.append(f'{rel}: expected {expected_size} bytes, got {actual_size}')
    if missing_or_bad:
        raise RuntimeError('Drive run sync verification failed:\n  - ' + '\n  - '.join(missing_or_bad))
    print('Drive run file-size verification passed for', len(file_sizes), 'file(s)')


def robust_copy_file(src: Path, dst_dir: Path, retries: int, sleep_sec: float) -> None:
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / src.name
    last_error = None
    for attempt in range(1, int(retries) + 1):
        try:
            shutil.copy2(src, dst)
            if dst.exists():
                print('Drive file sync verified:', dst)
                return
            last_error = RuntimeError(f'{dst} was not visible after copy')
        except Exception as exc:
            last_error = exc
        if attempt < int(retries):
            print(f'File sync failed; retrying in {sleep_sec} seconds:', last_error)
            time.sleep(float(sleep_sec))
    raise RuntimeError(f'Final Drive file sync failed after {retries} attempts: {last_error}')


if not SYNC_TO_DRIVE:
    print('Drive sync disabled with SOKE20_SYNC_TO_DRIVE=0')
elif same_resolved_path(OUTPUT_ROOT, DRIVE_SYNC_DEST):
    print('OUTPUT_ROOT is already the Drive sync destination:', OUTPUT_ROOT)
elif str(OUTPUT_ROOT).startswith('/content/drive/'):
    print('OUTPUT_ROOT is already on Google Drive:', OUTPUT_ROOT)
else:
    run_verify = 'best.pt' if (RUN_ROOT / 'best.pt').exists() else ('last.pt' if (RUN_ROOT / 'last.pt').exists() else None)
    if run_verify is not None:
        drive_run_dir = DRIVE_SYNC_DEST / 'runs' / RUN_ROOT.name
        robust_rsync(
            RUN_ROOT,
            drive_run_dir,
            DRIVE_SYNC_RETRIES,
            DRIVE_SYNC_RETRY_SLEEP_SEC,
            verify_rel=run_verify,
        )
        verify_synced_files(RUN_ROOT, drive_run_dir, run_file_sizes)
    else:
        print('No run checkpoint found to sync yet:', RUN_ROOT)

    if PREPROCESS_ROOT.exists():
        robust_rsync(
            PREPROCESS_ROOT,
            DRIVE_SYNC_DEST / 'preprocess_soke20',
            DRIVE_SYNC_RETRIES,
            DRIVE_SYNC_RETRY_SLEEP_SEC,
            verify_rel='train_manifest.csv',
            extra_args=['--exclude', 'cache_soke20/'],
        )

    robust_copy_file(status_path, DRIVE_SYNC_DEST, DRIVE_SYNC_RETRIES, DRIVE_SYNC_RETRY_SLEEP_SEC)


In [ ]:
# Final Colab cleanup. Disable with SOKE20_KEEP_RUNTIME=1 when you want to inspect the live runtime.
if os.environ.get('SOKE20_KEEP_RUNTIME', '0') != '1':
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print('Runtime unassign skipped:', exc)
else:
    print('Keeping runtime alive because SOKE20_KEEP_RUNTIME=1')
